In [1]:

import os
import sys
import json
import re
import numpy as np
import wikipediaapi


# プロジェクトのutils追加
project_root = os.path.join(os.getcwd(), "..")
sys.path.append(project_root)

# path = os.path.join(project_root, "data", "prompt", "base_for_gen_wikipedia_factual_relation_sentence_extraction_prompt.txt")
path = os.path.join(project_root, "data", "prompt", "base_for_gen_wikipedia_factual_relation_sentence_extraction_prompt_add_rels.txt")
with open(path, "r") as f:
    gen_prompt_base = f.read()


In [3]:

print_results = True  # get_concept_list_with_complete_cos_similar_termsの結果を表示するかどうか


def fetch_wikipedia_page(title: str, lang: str = "en") -> dict:
    """Fetch a Wikipedia page by title and language.
    wiki pageを取得する．
    Args:
        title (str): Wikipediaページのタイトル
        lang (str): Wikipediaの言語コード（例: "en", "ja"）

    Returns:
        dict: ページ情報を含む辞書．ページが存在しない場合は'exists'キーがFalseとなる．

    Note:
        * titleに完全一致するページのみが取得されるため，google検索のように最も近いが異なるものを検索結果として返すことはない．
        * そのため，提示した固有名詞とは違う情報のページが返されることは無い．
        * また，titleの綴りが間違っていても, userがよく間違う綴りの場合は, wikipedia側で正しいページにリダイレクトしてくれる．
            * 例: "Colloseum" -> "Colosseum"のページを取得
    
    """
    wiki = wikipediaapi.Wikipedia(
        user_agent="my-wikipedia-fetcher/1.0 (contact: your_email@example.com)",
        language=lang,
    )
    page = wiki.page(title)

    if not page.exists():
        return {"exists": False, "title": title, "lang": lang}

    return {
        "exists": True,
        "title": page.title,
        "url": page.fullurl,
        "summary": page.summary,
        "text": page.text,  # 全文（長いので注意）
    }


def extract_wiki_main_text(text: str) -> str:
    """Wikipediaページの情報から本文テキストのみを抽出する。
    Args:
        text (str): Wikipediaページの全文

    Returns:
        str: Wikipediaページの本文テキスト。ページが存在しない場合は空文字列を返す。
    """

    STOP_SECTIONS = {
        "see also",
        "references",
        "notes",
        "further reading",
        "external links",
        "bibliography",
        "sources",
        "works cited",
        "citations",
    }
    # 見出し候補を | で連結
    titles = "|".join(re.escape(x) for x in STOP_SECTIONS)

    # 行頭に単独で出る見出しを検出
    # 例:
    # See also
    # References
    #
    # または wiki風:
    # == See also ==
    pattern = rf"(?mi)^\s*(?:=+\s*)?(?:{titles})(?:\s*=+)?\s*$"

    # 最初に見つかった見出し以降を全て削除
    m = re.search(pattern, text)
    if m:
        text = text[:m.start()]

    # 脚注番号 [1], [23] を削除
    text = re.sub(r"\[\d+\]", "", text)

    # 空行を整理
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text



def main():

    wikiget_target_concepts = []

    print("=== Start Fetching Wikipedia Pages ===")
    # *** wikipediaページを取得して保存 ***
    wikipage_savedir = os.path.join(project_root, "data", "wikipages")
    os.makedirs(wikipage_savedir, exist_ok=True)

    # 既に保存されているwikipageを確認
    existing_wikipages = set()
    for filename in os.listdir(wikipage_savedir):
        if filename.endswith(".json"):
            concept = filename.split('.json')[0].replace('_', ' ')
            existing_wikipages.add(concept)

    # 各conceptに対してWikipediaページを取得し保存
    for concept in wikiget_target_concepts:
        # 既に保存されている場合はスキップ
        if concept in existing_wikipages:
            print(f"【Skipped】 Wikipedia page for '{concept}' already exists.")
            continue

        result = fetch_wikipedia_page(concept, lang="en")
        if not result["exists"]:
            print(f"【Skipped】 Wikipedia page for '{concept}' does not exist.")
            continue

        filename = f"{concept.replace(' ', '_')}.json"
        with open(os.path.join(wikipage_savedir, filename), "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
    
        print(f"【Saved】 Wikipedia page for '{concept}' saved as '{filename}'.")

    print("=== Finished Fetching Wikipedia Pages ===")


# if __name__ == "__main__":
#     main()

In [6]:
# target_concepts = ["Sieve of Eratosthenes", "quicksort",'Gaussian elimination', 'Monte Carlo method', "Dijkstra's algorithm"]
# target_concepts = ['honey', 'wheat', 'flour', 'yogurt', 'vegetarianism']
# target_concepts = ['Centaurea cyanus', 'evergreen plant', 'sugarcane', 'aquatic plant', 'succulent plant',]
target_concepts = ['ABBA', 'The Rolling Stones', 'Simon & Garfunkel', 't.A.T.u.',]

i = 0
i += 1
target_concept = target_concepts[i]

wiki_info = fetch_wikipedia_page(target_concept, lang="en")
main_text = extract_wiki_main_text(wiki_info['text'])
print(main_text)
wiki_info

The Rolling Stones are an English rock band formed in London in 1962. Active for over six decades, they are one of the most popular, influential, and enduring bands of the rock era. In the early 1960s, the band pioneered the gritty, rhythmically driven sound that came to define hard rock. Their first stable line-up consisted of vocalist Mick Jagger, guitarist Keith Richards, multi-instrumentalist Brian Jones, bassist Bill Wyman, and drummer Charlie Watts, after pianist Ian Stewart was side-lined by their manager Andrew Loog Oldham. During their early years, Jones was the primary leader. Oldham encouraged them to write their own songs. The Jagger–Richards partnership soon became the band's primary songwriting and creative force.
Rooted in blues and early rock and roll, the Rolling Stones started out playing cover versions and were at the forefront of the British Invasion in 1964, becoming identified with the youthful counterculture of the 1960s. They then found greater success with thei

{'exists': True,
 'title': 'The Rolling Stones',
 'url': 'https://en.wikipedia.org/wiki/The_Rolling_Stones',
 'summary': 'The Rolling Stones are an English rock band formed in London in 1962. Active for over six decades, they are one of the most popular, influential, and enduring bands of the rock era. In the early 1960s, the band pioneered the gritty, rhythmically driven sound that came to define hard rock. Their first stable line-up consisted of vocalist Mick Jagger, guitarist Keith Richards, multi-instrumentalist Brian Jones, bassist Bill Wyman, and drummer Charlie Watts, after pianist Ian Stewart was side-lined by their manager Andrew Loog Oldham. During their early years, Jones was the primary leader. Oldham encouraged them to write their own songs. The Jagger–Richards partnership soon became the band\'s primary songwriting and creative force.\nRooted in blues and early rock and roll, the Rolling Stones started out playing cover versions and were at the forefront of the British In

In [5]:

gen_prompt = gen_prompt_base.replace("<target_concept>", target_concept).replace("<wikipedia_text>", main_text)
print(gen_prompt)


You are given the full main text of a Wikipedia article describing a specific proper noun.

Your task is to extract factual characteristics of the target object using only the information explicitly stated or clearly implied in the provided Wikipedia text.

### Input

#### Target object
evergreen plant

#### Wikipedia article text
In botany, an evergreen is a plant which has foliage that remains green and functional throughout the year. This contrasts with deciduous plants, which lose their foliage completely during the winter or dry season. Consisting of many different species, the unique feature of evergreen plants lends itself to various environments and purposes.

Evergreen species
There are many different kinds of evergreen plants, including trees, shrubs, and vines. Evergreens include:

Most species of conifers (e.g., pine, hemlock, spruce, and fir), but not all (e.g., larch).
Live oak, holly, and "ancient" gymnosperms such as cycads
Many woody plants from frost-free climates
Ra

In [33]:
# text = wiki_info["text"]

# 本文中のconceptを<unused>に置換する. 小文字大文字は区別しない
# text = re.sub(r'\bSieve of Eratosthenes\b', '<unused>', main_text, flags=re.IGNORECASE)
text = re.sub(rf'\b{re.escape(target_concept)}\b', '<unused>', main_text, flags=re.IGNORECASE)
print(text)

<unused> is a group of wild and domesticated grasses of the genus Triticum (). As cereals, they are cultivated for their grains, which are staple foods around the world. Well-known <unused> species and hybrids include the most widely grown common <unused> (T. aestivum), spelt, durum, emmer, einkorn, and Khorasan or Kamut. The archaeological record suggests that <unused> was first cultivated in the regions of the Fertile Crescent around 9600 BC.
<unused> is grown on a larger area of land than any other food crop (220.7 million hectares or 545 million acres in 2021). World trade in <unused> is greater than that of all other crops combined. In 2021, world <unused> production was 771 million tonnes (850 million short tons), making it the second most-produced cereal after maize (known as corn in North America and Australia; <unused> is often called corn in other countries including Britain). Since 1960, world production of <unused> and other grain crops has tripled and is expected to grow f